<a href="https://colab.research.google.com/github/feriyanwyn/dokter-daun-tomat/blob/main/CNN_Dokter_Daun_Tomat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍅 Dokter Daun Tomat: CNN Pendeteksi Penyakit Tanaman
Tomat ditanam di mana-mana, dari kebun petani sampai pot di halaman rumah. Penyakit daun yang terlambat diketahui bisa merusak seluruh tanaman. Notebook ini melatih **CNN (transfer learning MobileNetV2)** untuk mengenali **9 penyakit daun tomat + daun sehat** dari sebuah foto.

**Yang kamu dapat:**
- Model yang mendiagnosis foto daun tomat, lengkap dengan **Grad-CAM** (bagian daun yang jadi dasar keputusan model).
- Fitur **upload foto daun sendiri** beserta tips penanganan umum dalam Bahasa Indonesia.
- Ekspor ke **TensorFlow Lite** (`.tflite`) supaya bisa dipakai di aplikasi HP.

**Dataset:** [PlantVillage](https://www.tensorflow.org/datasets/catalog/plant_village) (foto daun yang sudah berlabel), diunduh otomatis oleh notebook ini (±830 MB, hanya bagian daun tomat yang dipakai).

**Cara pakai:** Runtime → Change runtime type → **GPU (T4)**, lalu Run all. Total ±15–25 menit (termasuk unduh dataset).

## 1. Import library

In [ ]:
# Perbaikan: tfds di Python 3.13 butuh paket ini
!pip install -q importlib_resources

import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Kamus penyakit (nama Indonesia + tips penanganan umum)
Tips ini bersifat **umum** sebagai langkah awal. Untuk keputusan penggunaan pestisida atau penanganan serius, konsultasikan dengan penyuluh pertanian setempat.

In [ ]:
INFO = {
    'bacterial spot': ('Bercak bakteri',
        'Hindari menyiram dari atas (percikan air menyebarkan bakteri), buang daun yang terinfeksi, dan lakukan rotasi tanaman.'),
    'early blight': ('Hawar daun awal (Early blight)',
        'Buang daun bawah yang bergejala, beri mulsa agar tanah tidak terpercik ke daun, jaga jarak tanam, dan rotasi tanaman.'),
    'late blight': ('Hawar daun akhir / busuk daun (Late blight)',
        'Menyebar cepat saat lembap dan sejuk. Cabut dan musnahkan bagian/tanaman terinfeksi (jangan dijadikan kompos), segera hubungi penyuluh pertanian.'),
    'leaf mold': ('Jamur daun (Leaf mold)',
        'Kurangi kelembapan, perbaiki sirkulasi udara (pangkas daun yang terlalu rapat), dan hindari membasahi daun.'),
    'septoria': ('Bercak daun Septoria',
        'Buang daun terinfeksi, hindari percikan air ke daun, bersihkan sisa tanaman, dan rotasi tanaman.'),
    'spider mites': ('Tungau laba-laba',
        'Periksa bagian bawah daun, semprot air bertekanan lembut, jaga tanaman tidak terlalu kering/berdebu, dan pantau penyebarannya.'),
    'target spot': ('Bercak target (Target spot)',
        'Perbaiki sirkulasi udara, buang daun dan sisa tanaman terinfeksi, serta lakukan rotasi tanaman.'),
    'yellow leaf curl': ('Virus keriting kuning (TYLCV)',
        'Virus ini ditularkan kutu kebul (whitefly). Kendalikan kutu kebul, cabut dan musnahkan tanaman terinfeksi, gunakan bibit/varietas tahan.'),
    'mosaic': ('Virus mosaik',
        'Tidak ada obatnya. Cabut tanaman terinfeksi, cuci tangan dan sterilkan alat setelah memegangnya, dan gunakan benih sehat.'),
    'healthy': ('Sehat',
        'Daun terlihat sehat. Lanjutkan perawatan rutin dan tetap pantau tanaman secara berkala.'),
}

def clean_name(raw):
    return re.sub(r'^tomato[_\s-]*', '', raw, flags=re.I).replace('_', ' ').strip()

def lookup(raw):
    c = clean_name(raw).lower()
    for key, (nama, tips) in INFO.items():
        if key in c:
            return nama, tips
    return clean_name(raw), 'Konsultasikan dengan penyuluh pertanian.'

## 3. Unduh & siapkan dataset
Hanya daun tomat yang diambil (10 kelas), diubah ukurannya menjadi 160×160 piksel. Langkah ini membaca seluruh dataset sekali, jadi butuh beberapa menit.

In [ ]:
IMG = 160

ds, info = tfds.load('plant_village', split='train', as_supervised=True, with_info=True)
all_names = info.features['label'].names
tomato_ids = [i for i, n in enumerate(all_names) if 'tomato' in n.lower()]
print("Kelas tomat:", [all_names[i] for i in tomato_ids])

tomato_ids_t = tf.constant(tomato_ids, dtype=tf.int64)
ds_t = (ds
        .filter(lambda img, lab: tf.reduce_any(tf.equal(tf.cast(lab, tf.int64), tomato_ids_t)))
        .map(lambda img, lab: (tf.saturate_cast(tf.round(tf.image.resize(img, (IMG, IMG))), tf.uint8), lab)))

old2new = {old: new for new, old in enumerate(tomato_ids)}
images, labels = [], []
for img, lab in tfds.as_numpy(ds_t):
    images.append(img)
    labels.append(old2new[int(lab)])

X = np.stack(images)
y = np.array(labels)
raw_names = [all_names[i] for i in tomato_ids]
nama_id = [lookup(n)[0] for n in raw_names]
tips_id = [lookup(n)[1] for n in raw_names]
NUM_CLASSES = len(nama_id)
print("Total foto daun tomat:", X.shape, "| Jumlah kelas:", NUM_CLASSES)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/plant_village/incomplete.M2MGA0_1.0.2/plant_village-train.tfrecord-[0-9][0…

Dataset plant_village downloaded and prepared to /root/tensorflow_datasets/plant_village/1.0.2. Subsequent calls will reuse this data.
Kelas tomat: ['Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___healthy', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___Tomato_mosaic_virus', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus']
Total foto daun tomat: (18159, 160, 160, 3) | Jumlah kelas: 10


### Sebaran data & contoh foto tiap kelas

In [ ]:
counts = np.bincount(y, minlength=NUM_CLASSES)
plt.figure(figsize=(9, 4.5))
plt.barh(nama_id, counts, color='tomato')
plt.xlabel('Jumlah foto'); plt.title('Sebaran data per kelas'); plt.gca().invert_yaxis()
plt.tight_layout(); plt.show()

plt.figure(figsize=(15, 6.5))
for c in range(NUM_CLASSES):
    plt.subplot(2, 5, c + 1)
    plt.imshow(X[np.where(y == c)[0][0]])
    plt.title(nama_id[c], fontsize=9); plt.axis('off')
plt.tight_layout(); plt.show()

### (Opsional) Simpan dataset daun tomat sebagai file ZIP
Menyimpan seluruh foto daun tomat (sudah diubah ke 160×160 piksel) ke folder per kelas, lalu di-zip. Strukturnya:
`dataset_daun_tomat/<nama_kelas>/daun_00001.jpg`, siap dipakai ulang dengan `image_dataset_from_directory`.

Ukurannya bisa mencapai ratusan MB. Jika unduhan lewat browser gagal, ubah `KE_DRIVE = True` agar ZIP disalin ke Google Drive.


In [ ]:
import os, shutil
from google.colab import files

KE_DRIVE = False        # True = salin ZIP ke Google Drive, bukan unduh lewat browser
root = 'dataset_daun_tomat'

folders = [clean_name(n).replace(' ', '_') for n in raw_names]
for fo in folders:
    os.makedirs(f"{root}/{fo}", exist_ok=True)

for i, (im, lab) in enumerate(zip(X, y)):
    Image.fromarray(im).save(f"{root}/{folders[lab]}/daun_{i:05d}.jpg", quality=92)

shutil.make_archive(root, 'zip', root)
print(f"Dataset tersimpan: {root}.zip ({os.path.getsize(root + '.zip') / 1024 / 1024:.0f} MB)")

if KE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copy(f'{root}.zip', '/content/drive/MyDrive/')
    print("ZIP disalin ke Google Drive (MyDrive).")
else:
    files.download(f'{root}.zip')

Dataset tersimpan: dataset_daun_tomat.zip (194 MB)


ValueError: mount failed

## 4. Bagi data: train / validasi / test
Pembagian **stratified** (proporsi tiap kelas dijaga). Data tidak seimbang (misalnya virus mosaik jauh lebih sedikit), maka dipakai **class weight** agar kelas kecil tidak diabaikan.

In [ ]:
x_trainval, x_test, y_trainval, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
x_train, x_val, y_train, y_val = train_test_split(x_trainval, y_trainval, test_size=0.1765, stratify=y_trainval, random_state=42)
print("Train:", len(x_train), "| Val:", len(x_val), "| Test:", len(x_test))

cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
class_weight = {i: float(w) for i, w in enumerate(cw)}

BATCH = 64

def augment(img, lab):
    img = tf.cast(img, tf.float32)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 25.0)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    return tf.clip_by_value(img, 0.0, 255.0), lab

def make_ds(x_arr, y_arr, train=False):
    d = tf.data.Dataset.from_tensor_slices((x_arr, y_arr))
    if train:
        d = d.shuffle(4000, seed=42).map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        d = d.map(lambda i, l: (tf.cast(i, tf.float32), l), num_parallel_calls=tf.data.AUTOTUNE)
    return d.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(x_train, y_train, train=True)
val_ds   = make_ds(x_val, y_val)
test_ds  = make_ds(x_test, y_test)

Train: 12710 | Val: 2725 | Test: 2724


## 5. Bangun model: MobileNetV2 (pretrained ImageNet) + kepala klasifikasi
Model CNN yang sudah belajar mengenali tepi, tekstur, dan pola dari jutaan foto (ImageNet) dipakai ulang (*transfer learning*), lalu hanya bagian akhirnya dilatih untuk mengenali penyakit daun tomat. Cara ini jauh lebih cepat dan akurat dibanding melatih dari nol.

In [ ]:
base = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')
base.trainable = False

head = keras.Sequential([
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.25),
    layers.Dense(NUM_CLASSES, activation='softmax'),
], name='head')

inputs = keras.Input(shape=(IMG, IMG, 3))
x = layers.Rescaling(1 / 127.5, offset=-1)(inputs)    # skala 0-255 -> -1..1 (sesuai MobileNetV2)
x = base(x, training=False)
outputs = head(x)
model = keras.Model(inputs, outputs, name='dokter_daun_tomat')

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 6. Training (2 tahap)
**Tahap 1:** hanya melatih kepala klasifikasi. **Tahap 2 (fine-tuning):** sebagian layer akhir MobileNetV2 ikut dilatih dengan learning rate kecil.

In [ ]:
cb = lambda: [keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]

print("=== Tahap 1: melatih kepala klasifikasi ===")
h1 = model.fit(train_ds, validation_data=val_ds, epochs=6,
               class_weight=class_weight, callbacks=cb())

In [ ]:
print("=== Tahap 2: fine-tuning 40 layer terakhir ===")
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False
for layer in base.layers[-40:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(3e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=10,
               class_weight=class_weight, callbacks=cb())

acc  = h1.history['accuracy'] + h2.history['accuracy']
vacc = h1.history['val_accuracy'] + h2.history['val_accuracy']
loss = h1.history['loss'] + h2.history['loss']
vloss = h1.history['val_loss'] + h2.history['val_loss']
n1 = len(h1.history['loss'])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(acc, label='train'); ax[0].plot(vacc, label='validasi'); ax[0].axvline(n1 - 0.5, ls='--', c='gray')
ax[0].set_title('Akurasi (garis putus = mulai fine-tuning)'); ax[0].legend(); ax[0].set_xlabel('Epoch')
ax[1].plot(loss, label='train'); ax[1].plot(vloss, label='validasi'); ax[1].axvline(n1 - 0.5, ls='--', c='gray')
ax[1].set_title('Loss'); ax[1].legend(); ax[1].set_xlabel('Epoch')
plt.show()

## 7. Evaluasi di data test
Data test **tidak pernah dilihat** model selama training.

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Akurasi test: {test_acc*100:.2f}%\n")

probs = model.predict(test_ds, verbose=0)
y_pred = probs.argmax(1)
print(classification_report(y_test, y_pred, target_names=nama_id, digits=3))

cm = confusion_matrix(y_test, y_pred, normalize='true')
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Reds', xticklabels=nama_id, yticklabels=nama_id)
plt.xlabel('Prediksi'); plt.ylabel('Sebenarnya'); plt.title('Confusion matrix (dinormalisasi per kelas)')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

## 8. 🔍 Grad-CAM: model melihat bagian mana dari daun?
Area **merah/kuning** adalah bagian yang paling memengaruhi keputusan model. Model yang baik seharusnya fokus ke **bercak atau kerusakan pada daun**, bukan ke latar belakang.

In [ ]:
def gradcam(img_uint8):
    xin = tf.cast(img_uint8[np.newaxis], tf.float32) / 127.5 - 1.0
    with tf.GradientTape() as tape:
        conv_out = base(xin, training=False)
        preds = head(conv_out, training=False)
        idx = int(tf.argmax(preds[0]))
        score = preds[:, idx]
    grads = tape.gradient(score, conv_out)
    w = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv_out[0] * w, axis=-1))
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., tf.newaxis], (IMG, IMG))[..., 0].numpy()
    return cam, idx, preds[0].numpy()

def overlay(img_uint8, cam):
    heat = plt.cm.jet(cam)[..., :3]
    return np.clip(0.6 * (img_uint8 / 255.0) + 0.4 * heat, 0, 1)

rng = np.random.default_rng(7)
sel = rng.choice(len(x_test), 6, replace=False)
fig, axes = plt.subplots(2, 6, figsize=(17, 6))
for col, i in enumerate(sel):
    cam, idx, p = gradcam(x_test[i])
    benar = idx == y_test[i]
    axes[0, col].imshow(x_test[i]); axes[0, col].set_title(f"Asli: {nama_id[y_test[i]]}", fontsize=8)
    axes[1, col].imshow(overlay(x_test[i], cam))
    axes[1, col].set_title(f"{'✅' if benar else '❌'} {nama_id[idx]}\n({p[idx]*100:.0f}%)", fontsize=8,
                           color='green' if benar else 'red')
    axes[0, col].axis('off'); axes[1, col].axis('off')
plt.tight_layout(); plt.show()

## 9. Kasus yang salah ditebak
Melihat kesalahan membantu memahami batas kemampuan model.

In [ ]:
salah = np.where(y_pred != y_test)[0]
print(f"Jumlah salah: {len(salah)} dari {len(y_test)}")

if len(salah) > 0:
    show = salah[:6]
    fig, axes = plt.subplots(2, len(show), figsize=(2.8 * len(show), 6))
    axes = np.atleast_2d(axes).reshape(2, -1)
    for col, i in enumerate(show):
        cam, idx, p = gradcam(x_test[i])
        axes[0, col].imshow(x_test[i]); axes[0, col].set_title(f"Asli: {nama_id[y_test[i]]}", fontsize=8)
        axes[1, col].imshow(overlay(x_test[i], cam)); axes[1, col].set_title(f"Tebak: {nama_id[idx]}", fontsize=8, color='red')
        axes[0, col].axis('off'); axes[1, col].axis('off')
    plt.tight_layout(); plt.show()

## 10. 📷 Diagnosis foto daun tomat milikmu
Upload foto daun tomat. **Tips foto yang baik:** satu daun saja, dekat, cahaya cukup, dan latar polos (karena dataset latihnya berlatar polos).

⚠️ Model hanya mengenal 10 kondisi di atas. Jika kamu mengunggah foto yang bukan daun tomat, model tetap akan memilih salah satu jawaban, jadi perhatikan juga tingkat keyakinannya.

In [ ]:
from google.colab import files

uploaded = files.upload()
for name, data in uploaded.items():
    import io
    img = ImageOps.fit(Image.open(io.BytesIO(data)).convert('RGB'), (IMG, IMG))
    arr = np.array(img, dtype=np.uint8)

    cam, idx, p = gradcam(arr)
    top3 = np.argsort(p)[::-1][:3]

    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(arr); ax[0].set_title(name, fontsize=9); ax[0].axis('off')
    ax[1].imshow(overlay(arr, cam)); ax[1].set_title('Grad-CAM', fontsize=9); ax[1].axis('off')
    plt.show()

    print("Hasil diagnosis:")
    for i in top3:
        print(f"  {nama_id[i]:45s} {p[i]*100:5.1f}%")
    print("\nSaran awal:", tips_id[idx])
    if p[idx] < 0.70:
        print("\n⚠️ Model kurang yakin. Coba foto ulang (satu daun, dekat, cahaya cukup, latar polos) atau tanyakan ke penyuluh pertanian.")
    print("-" * 70)

## 11. 📱 Ekspor model untuk aplikasi HP (TensorFlow Lite)
File `.tflite` kecil dan ringan, cocok dipakai di aplikasi Android/iOS. **Input:** gambar 160×160×3 dengan nilai piksel 0–255 (float32). **Output:** 10 probabilitas sesuai urutan `labels.txt`.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('daun_tomat.tflite', 'wb') as f:
    f.write(tflite_model)
with open('labels.txt', 'w') as f:
    f.write("\n".join(nama_id))
model.save('daun_tomat.keras')

print(f"Ukuran model TFLite: {len(tflite_model)/1024/1024:.1f} MB")
files.download('daun_tomat.tflite')
files.download('labels.txt')

---
## ⚠️ Catatan penting & batasan
- **Ini alat bantu, bukan pengganti ahli.** Untuk keputusan penanganan serius (misalnya pestisida), konsultasikan dengan penyuluh pertanian.
- Foto PlantVillage umumnya **daun tunggal berlatar polos dan difoto dalam kondisi terkontrol**. Pada foto di kebun (latar rumit, cahaya berubah, banyak daun), akurasi biasanya **lebih rendah** daripada angka di atas.
- Dataset ini berisi beberapa foto dari daun yang sama pada sudut berbeda, sehingga akurasi test bisa terlihat lebih tinggi dari performa di dunia nyata.
- Model hanya mengenal **10 kondisi** yang ada di dataset. Penyakit lain atau kekurangan nutrisi bisa salah dikenali.

### 💡 Ide pengembangan
- Kumpulkan foto daun dari kebun sendiri (latar nyata) untuk **fine-tuning** lanjutan.
- Tambahkan tanaman lain dari PlantVillage (cabai, kentang, jagung) dan lihat apakah modelnya tetap akurat.
- Bungkus model `.tflite` ke aplikasi Android sederhana atau bot Telegram.